## Automated Detection Pipeline for PRIME

In [1]:
import candidate_sandbox
import generate_image_list
import sfft_util
import bogus
import ML_utils
import photometrus_utils as util


import pandas as pd
import torch


/home/alex/miniconda3/envs/photo310/lib/python3.10/site-packages/sfft/utils/HoughDetection.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


### 1. Identify and reduce transient targets

In [2]:
tns_file = 'data_results/single_example.csv' # Your list of transients from TNS
# combo_out_file = candidate_sandbox.identify_observations(tns_file) 

combo_out_file = '/home/alex/PycharmProjects/prime-photometry-fiona/automated_detection/data_results/new_combo_file'

In [3]:
good_candidates_combo = candidate_sandbox.select_candidates(tns_csv=tns_file, combo_file=combo_out_file)

1 relevant transients in TNS

10 identified target fields overlapping with these detections
1 obs within a 20 days of a TNS obs
1 obs after 2023
1 obs in sparse locations
1 obs with a ref of matching field and band
(0 solo obs)
0 new obs,  1 old obs, 1 total
wrote 4 good transients to file /home/alex/PycharmProjects/prime-photometry-fiona/automated_detection/data_results/new_combo_file-top-candidates


In [4]:
# candidate_sandbox.reduce_combo(good_candidates_combo)

### 2. Iterate through processed data, generate real and bogus triplets

In [5]:
candidates, obs_df, folders = generate_image_list.fetch_prime_data(transient_file = tns_file)
candidates, triplet_df, sci_df = generate_image_list.make_triplet_df(candidates, obs_df, folders)

0    10/19/25
Name: discoverydate, dtype: object
['AT2025aayv']
found /mnt/photometry/AT2025aayv/field8786-2025-11-03/J/stack/coadd.Open-J.03159535-03159571.C1.fits
added row!
found /mnt/photometry/AT2025aayv/field8785-2025-11-09/J/stack/coadd.Open-J.03172143-03172179.C2.fits
non detection (no ecsv):
added row!
found /mnt/photometry/AT2025aayv/field8786-2026-01-11/J/stack/coadd.Open-J.03362257-03362293.C1.fits
added row!
folders: 4

finding single detections...
0 detections with (id, field, and band) which appear once combo commands from 1/28
0 total triplets
0 detections after 2023
0 detections within 20 day window of ZTF measurement
Number of sci only fields that exist: 0
1 total triplets
1 detections after 2023
1 detections within 20 day window of ZTF measurement
distinct transients (excluding repeats with different bands, fields): 1

processed images: triplets: 1, only sci: 0, only ref: 1, partially processed: 2


/home/alex/PycharmProjects/prime-photometry-fiona/automated_detection/generate_image_list.py:412: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  obs_df = (obs_df.groupby(candidate_groups, group_keys=False).apply(assign_roles))


In [ ]:

# candidates = candidates[candidates['full_name'].isin(['AT2025aayv'])]

records = []
for row in triplet_df.itertuples():

    sci, ref = row.coadd_path, row.ref_coadd_path[row.ref_idx]
    FITS_SCI, FITS_REF, FITS_DIFF, PixA_DIFF, SFFTPrepDict =  sfft_util.SFFT(sci, ref)
    
    rms_ref, rms_sci, rms_diff, fwhm_sci, fwhm_ref, offset =  sfft_util.get_diff_stats(SFFTPrepDict, PixA_DIFF)
    min_distance, cat_file = util.distance_to_source(FITS_DIFF, row.ra, row.declination)

    
    diff_info = {'sfft_diff': FITS_DIFF,'diff_cat': cat_file, 'distance': min_distance,
                 'sci_rms': rms_sci, 'ref_rms': rms_ref, 'diff_rms':rms_diff, 
                 'sci_fwhm': fwhm_sci, 'ref_fwhm': fwhm_ref, 'offset': offset}
    
    record = {**row._asdict(), **diff_info}
    records.append(record)

df = pd.DataFrame(records)



/mnt/photometry/AT2025aayv
Sextracting base epoch: coadd.Open-J.03159535-03159571.C1.fits...
Including weight map!
Sextracting matching epoch: coadd.Open-J.03362257-03362293.C1.abs_astr.fits...


> 
----- SCAMP 2.14.0 started on 2026-09-17 at 15:53:42 with 20 threads

> 
----- 1 input:
> Examining Catalog coadd.Open-J.03362257-03362293.C1.abs_astr.cat
coadd.Open-J.03362257-03362293.C1.abs_astr.cat:  "no ident           "  no ext. header   1 set    4966 detections

----- 4966 detections loaded
> Grouping fields on the sky ...
> Grouping fields: field 1/1, 0 group
> 

----- 1 instrument found for astrometry:

Instrument A1 :
1 extension

----- 1 instrument found for photometry:

Instrument P1 :

----- 1 field group found:

 Group  1: 1 field at 04:18:28.42 -26:11:49.7 with radius 27.18'
                  instruments  epoch      center coordinates     radius   scale 
coadd.Open-J.0336225 A1  P1       0.0  04:18:28.42 -26:11:49.7   27.18'  0.4974"

> Making mosaic adjustments...
> 
----- Reference catalogs:

> Examining Catalog coadd.Open-J.03159535-03159571.C1.cat...
> Loading Catalog coadd.Open-J.03159535-03159571.C1.cat...

> WARNING: ref_epoch parameter not found in catalog coa

Sextracting base epoch: coadd.Open-J.03159535-03159571.C1.fits...
Including weight map!
Sextracting matching epoch: coadd.Open-J.03362257-03362293.C1.abs_astr.fits...


> 
----- SCAMP 2.14.0 started on 2026-09-17 at 15:53:46 with 20 threads

> 
----- 1 input:
> Examining Catalog coadd.Open-J.03362257-03362293.C1.abs_astr.cat
coadd.Open-J.03362257-03362293.C1.abs_astr.cat:  "no ident           "  no ext. header   1 set    4966 detections

----- 4966 detections loaded
> Grouping fields on the sky ...
> Grouping fields: field 1/1, 0 group
> 

----- 1 instrument found for astrometry:

Instrument A1 :
1 extension

----- 1 instrument found for photometry:

Instrument P1 :

----- 1 field group found:

 Group  1: 1 field at 04:18:28.42 -26:11:49.7 with radius 27.18'
                  instruments  epoch      center coordinates     radius   scale 
coadd.Open-J.0336225 A1  P1       0.0  04:18:28.42 -26:11:49.7   27.18'  0.4974"

> Making mosaic adjustments...
> 
----- Reference catalogs:

> Examining Catalog coadd.Open-J.03159535-03159571.C1.cat...
> Loading Catalog coadd.Open-J.03159535-03159571.C1.cat...

> WARNING: ref_epoch parameter not found in catalog coa

Sextracting base epoch: coadd.Open-J.03159535-03159571.C1.fits...
Including weight map!
Sextracting matching epoch: coadd.Open-J.03362257-03362293.C1.abs_astr.fits...


...

> 
----- SCAMP 2.14.0 started on 2026-09-17 at 15:53:56 with 20 threads

> 
----- 1 input:
> Examining Catalog coadd.Open-J.03362257-03362293.C1.abs_astr.cat
coadd.Open-J.03362257-03362293.C1.abs_astr.cat:  "no ident           "  no ext. header   1 set    4966 detections

----- 4966 detections loaded
> Grouping fields on the sky ...
> Grouping fields: field 1/1, 0 group
> 

----- 1 instrument found for astrometry:

Instrument A1 :
1 extension

----- 1 instrument found for photometry:

Instrument P1 :

----- 1 field group found:

 Group  1: 1 field at 04:18:28.42 -26:11:49.7 with radius 27.19'
                  instruments  epoch      center coordinates     radius   scale 
coadd.Open-J.0336225 A1  P1       0.0  04:18:28.42 -26:11:49.7   27.19'  0.4974"

> Making mosaic adjustments...
> 
----- Reference catalogs:

> Examining Catalog coadd.Open-J.03159535-03159571.C1.cat...
> Loading Catalog coadd.Open-J.03159535-03159571.C1.cat...

> WARNING: ref_epoch parameter not found in catalog coa

Sextracting base epoch: coadd.Open-J.03159535-03159571.C1.fits...
Including weight map!
Sextracting matching epoch: coadd.Open-J.03362257-03362293.C1.abs_astr.fits...


> 
----- SCAMP 2.14.0 started on 2026-09-17 at 15:54:00 with 20 threads

> 
----- 1 input:
> Examining Catalog coadd.Open-J.03362257-03362293.C1.abs_astr.cat
coadd.Open-J.03362257-03362293.C1.abs_astr.cat:  "no ident           "  no ext. header   1 set    4966 detections

----- 4966 detections loaded
> Grouping fields on the sky ...
> Grouping fields: field 1/1, 0 group
> 

----- 1 instrument found for astrometry:

Instrument A1 :
1 extension

----- 1 instrument found for photometry:

Instrument P1 :

----- 1 field group found:

 Group  1: 1 field at 04:18:28.42 -26:11:49.7 with radius 27.19'
                  instruments  epoch      center coordinates     radius   scale 
coadd.Open-J.0336225 A1  P1       0.0  04:18:28.42 -26:11:49.7   27.19'  0.4974"

> Making mosaic adjustments...
> 
----- Reference catalogs:

> Examining Catalog coadd.Open-J.03159535-03159571.C1.cat...
> Loading Catalog coadd.Open-J.03159535-03159571.C1.cat...

> WARNING: ref_epoch parameter not found in catalog coa

MeLOn CheckPoint: Run SWarp Command ... 
 cd /tmp/PYSWarp_e8_etfuf && swarp /mnt/photometry/AT2025aayv/field8786-2026-01-11/J/stack/coadd.Open-J.03362257-03362293.C1.abs_astr.fits -IMAGEOUT_NAME /tmp/PYSWarp_e8_etfuf/coadd.Open-J.03362257-03362293.C1.abs_astr.tmp_resamp.fits -WEIGHTOUT_NAME /tmp/PYSWarp_e8_etfuf/coadd.Open-J.03362257-03362293.C1.abs_astr.tmp_resamp_weight.fits -c /tmp/PYSWarp_e8_etfuf/PYSWarp_gpwzvsofswarp_config/PYSWarp.swarp


> Setting up background map at line:   1024 / 4613   
> Setting up background map at line:   1152 / 4613   
> Setting up background map at line:   1280 / 4613   
> Setting up background map at line:   1408 / 4613   
> Setting up background map at line:   1536 / 4613   
> Setting up background map at line:   1664 / 4613   
> Setting up background map at line:   1792 / 4613   
> Setting up background map at line:   1920 / 4613   
> Setting up background map at line:   2048 / 4613   
> Setting up background map at line:   2176 / 4613   
> Setting up background map at line:   2304 / 4613   
> Setting up background map at line:   2432 / 4613   
> Setting up background map at line:   2560 / 4613   
> Setting up background map at line:   2688 / 4613   
> Setting up background map at line:   2816 / 4613   
> Setting up background map at line:   2944 / 4613   
> Setting up background map at line:   3072 / 4613   
> Setting up background map at line:   3200 / 4613   
> Setting up background map 

In [ ]:
full_data_path = "/home/alex/PycharmProjects/prime-photometry-fiona/automated_detection/data_results/full_data.csv"
df.to_csv(full_data_path)
full_df = bogus.make_df(csv = full_data_path)
# "/home/alex/PycharmProjects/prime-photometry-fiona/automated_detection/data_results/triplets_diff_detected.csv"

### 3. Machine Learning

In [ ]:
print(full_df.keys())
full_df.head()
triplet_df = ML_utils.ML_prep(full_df.copy())


In [ ]:
X_cols = ['SNR', 'Elongation', 'sci_rms', 'ref_rms', 'sci_cat_len', 'ref_cat_len', 
                        'sci_mag', 'diff_mag', 'n_neg', 'autoMag', 'aperMag', 'psfMag', 'b', 'psf_minus_aper']

metadata= triplet_df.copy()
X = metadata[X_cols].values
y = metadata['bogus'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

rf_model = RandomForestRegressor(max_depth=4,
                                n_estimators=100, 
                                random_state=42,
                                 max_features="sqrt")
rf_model.fit(X_train, y_train)

ypred_train = rf_model.predict(X_train)
ypred_test = rf_model.predict(X_test)

print("test mse: ", mse(y_test, ypred_test), "\ntrain mse: ", mse(y_train, ypred_train))
print("\ntest r2", rf_model.score(X_test, y_test), "\ntrain r2: ", rf_model.score(X_train, y_train))
print("Depths:", [estimator.tree_.max_depth for estimator in rf_model.estimators_])


### 4. Evaluation

In [ ]:
# test on one coadd
# make dashboard
